<div dir="rtl"><h2>05 — שערים צפויים: מידע מוקדם נכנס, שני ערכי λ יוצאים</h2>
<p>EXP-005 משתמש בעונות הפיתוח בלבד. D3, ההרצה המלאה של סולם הניסוי, הושלם. התוצאות בפרק 6 הן טיוטה; ההכרעות ממתינות לעמית ול-QA (בדיקת איכות עצמאית של Claude). המחברת קוראת תוצרים חתומים ואינה מאמנת מודל.</p><p>as_of הוא בוקר החיזוי: כל פיצ׳ר משתמש רק במה שקדם לו. λ הוא ממוצע השערים הצפוי לקבוצה, ולא תוצאה מובטחת.</p></div>

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config/exp005.yaml").exists():
    if root == root.parent:
        raise RuntimeError("Run from the project checkout")
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import pandas as pd
import plotly.io as pio
from IPython.display import HTML, display

from ligat.viz import goals_notebook as views
from ligat.viz import goals_walk as walk
from ligat.viz.presentation import display_table

pio.renderers.default = "plotly_mimetype"
data = views.development_data()


def table(frame, names, **kwargs):
    kwargs.setdefault("unicode_minus", list(frame.select_dtypes(include="number").columns))
    display(display_table(frame.rename_axis(None), names, columns=list(frame.columns), **kwargs))


def show(figure, name):
    figure.show()


display(HTML(walk.input_summary(data)))
display(HTML(walk.provisional_summary(data)))

<div dir="rtl"><h2>0 — ממתין ל-QA ולהכרעת עמית</h2>
<p>כאן יופיעו הפיצ׳רים שנכנסו, הדיוק, ומה עובר ל-M3. טרם נבחר מודל וטרם נכתבה שכבת λ אמיתית. גם כאשר D3 מסתיים, סימוני כלל 6 נשארים להכרעה אנושית.</p>
</div>

<div dir="rtl"><h2>1 — M2 חוזה שערים לכל קבוצה בבוקר המשחק</h2>
<p>λ הוא ממוצע חיובי של שערים צפויים, לא תוצאה מובטחת ולא הסתברות לניצחון. למשל λ=1.6 אומר 1.6 שערים בממוצע במפגשים חוזרים באותם תנאים; משחק יחיד עדיין יכול להסתיים באפס או בארבעה. M3 מקבל שני λ ובונה מטריצת תוצאות; הסימולציה מתרגמת משחקים לטבלה. התרשים מתאר חוזה נתונים, ללא אומדן או רווח סמך.</p>
</div>

In [ ]:
display(HTML(walk.chain_html()))

<div dir="rtl"><h3>1.1 — המודלים שמשתתפים</h3>
<p><b>GLM של Poisson:</b> כל פיצ׳ר מכפיל את קצב השערים בגורם קבוע; למשל ביתיות של פי 1.2 היא דוגמה כללית. מעט פרמטרים נותנים מודל יציב שקל להסביר בכדורגל. EXP-002 הוא סרגל GLM.</p>
<p><b>XGBoost:</b> הרבה עצי החלטה קטנים; כל עץ מתקן מעט את טעות הקודמים (boosting). הוא לומד קשרים לא ליניאריים ואינטראקציות, כמו שילוב ביתיות וכוח יריבה, אך במעט נתונים עלול לשנן.</p>
<p><b>CatBoost:</b> משפחה דומה של עצים עם טיפול מובנה בקטגוריות, למשל שלב הליגה. משמש להשוואה לשיטת העצים האחרת.</p>
<p><b>Maher בסגנון Dixon-Coles:</b> לומד כוח התקפה והגנה לכל קבוצה ויתרון בית. מעדכן את הכוחות לפני כל יום משחקים, כך שניצחונות עבר יכולים לשנות את ציפיית השערים הבאה. הנוסחאות והפרטים בפרק 3.</p></div>

<div dir="rtl"><h2>2.1 — כל משחק תורם שתי שורות, עם יעד משלו</h2>
<p>שורה היא קבוצה במשחק. היעד goals_scored הוא מספר השערים באותו משחק, מתוך matches; goals_for של prematch הוא סכום עבר ואינו היעד. הקלט הוא prematch_v2 הקפוא. ספירת השורות היא מפקד מלא של הנתונים, ולכן אין לה רווח סמך.</p>
</div>

In [ ]:
show(walk.season_games_figure(data), "E3a_season_games")
display(HTML(walk.warmup_html(data)))

<div dir="rtl"><h2>2.2 — בכל קיפול, ההיסטוריה קודמת לעונת המבחן</h2>
<p>Walk-forward מאמן על העבר ובודק על העונה הבאה. עונות החימום אינן משמשות ולידציה; יש 15 קיפולים. מודלי העצים ו-GLM מתאמנים לקיפול. Maher מתעדכן לפני כל יום משחקים, וגם בו משתמשים רק בתוצאות שקדמו לתחילת היום. הצבע מתאר תוכנית קבועה, ולא שיעור שדורש רווח סמך.</p>
</div>

In [ ]:
show(walk.folds_figure(data), "E3a_walk_forward")

<div dir="rtl"><h2>2.3 — זמן מחצית קצר מעדיף את הכדורגל העדכני</h2>
<p>דעיכה היא כמה משחק ישן נספר באימון. ציר x הוא כמה עונות אחורה המשחק נמצא, ציר y הוא המשקל שלו, וכל קו הוא זמן מחצית אחר. זמן מחצית h הוא מספר העונות שאחריהן משחק נספר כחצי.</p>

</div>

$$
w(a;h)=2^{-a/h},\qquad a=s_{validation}-1-s_{training},\qquad w=1\text{ ללא דעיכה}.
$$

<div dir="rtl">
<p>
<p>המרחק נמדד מעונת האימון האחרונה, שמשקלה 1; הטווח הוא מעל 0 ועד 1. כשחוזים את 2023/24 עם h=4, העונה 2018/19 נמצאת ארבע עונות לפני עונת האימון האחרונה ולכן משקלה חצי; עם h=1 משקלה 1/16. בלי דעיכה, כל העונות שוות. המשקל אינו מנורמל מחדש. אלו עקומות מתמטיות מדויקות, ללא אי-ודאות מדגמית.</p><p>ב-Maher אותו עיקרון, אך המרחק הוא הפרש הימים חלקי 365.25. הגרף מציג ימים ובעונות.</p></p></div>

In [ ]:
show(views.decay_figure(), "E2a_season_decay")
show(walk.daily_decay_figure(), "E2a_daily_decay")
display(HTML(walk.decay_result_html()))

<div dir="rtl"><h2>3.1 — Poisson מתאר ספירת שערים, ו-NLL מודד הפתעה</h2>
<p>למספר שערים שלם </p></div>

$$
Y\ge0
$$

<div dir="rtl"> ולממוצע </div>

$$
\lambda>0
$$

<div dir="rtl">:
</div>

$$
P(Y=y\mid\lambda)=e^{-\lambda}\lambda^y/y!,\qquad \ell(y,\lambda)=\lambda-y\log\lambda+\log\Gamma(y+1).
$$

<div dir="rtl">
<p>הסתברות בטווח [0,1]; NLL בטווח [0,∞), וערך נמוך יותר עדיף. למשל שער אחד עם λ=1.6 נותן NLL≈1.130. מניחים שוויון בין הממוצע לשונות של Poisson; זו הנחה שנבדקת בכיול. Poisson deviance הוא </p></div>

$$
2[y\log(y/\lambda)-(y-\lambda)]
$$

<div dir="rtl"> (איבר האפס מוגדר אפס); הוא מחסיר קבוע של יעד לצורכי אימון, ולכן אין להשוות אותו ישירות ל-NLL גולמי. רגולריזציה משפיעה על האימון, אך לא מתווספת למדד הוולידציה.
<p>בדוגמה הבאה λ=1.6 נבחר להמחשה בלבד, ולא נאמד או נבחר לפי משחק זה. התוצאה נלקחה ממשחק אמיתי. העמודות הן הסתברויות מדויקות במודל; n=1 משחק להמחשה, בלי הסקה סטטיסטית.</p>
</div>

In [ ]:
figure, example = views.poisson_example(data)
table(
    example,
    {
        "game_id": "מזהה משחק",
        "team": "קבוצה",
        "opponent": "יריבה",
        "season": "עונה",
        "observed": "שערים בפועל",
        "lambda_example": "λ להמחשה",
        "nll_example": "NLL להמחשה",
    },
    precision={"nll_example": 4},
)
show(figure, "E2a_poisson_example")

<div dir="rtl"><h2>3.2 — XGBoost מוסיף עצים שמתקנים את חיזוי השערים</h2>
</div>

$$
\lambda(x)=\exp\left[b+\sum_{t=1}^{T}\eta f_t(x)\right].
$$

<div dir="rtl">
<p>כל עץ לומד תיקון קטן משילוב פיצ׳רים, למשל פער Elo וביתיות. האקספוננט מבטיח λ חיובי. count:poisson הוא יעד האימון, hist היא שיטת הבנייה. CatBoost משתמש גם הוא בעצים ובהפסד Poisson, עם טיפול משותף בקטגוריות. stage מקודד כן/לא ברשת; מזהי קבוצה ויריבה נבדקים בנפרד בסוף הסולם, עם ערך ״לא מוכרת״ שנקבע מההיסטוריה בלבד.</p>
</div>

<div dir="rtl"><h2>3.3 — GLM נותן בסיס פשוט, Maher מפריד התקפה והגנה</h2>
</div>

$$
\lambda_i=\exp(\beta_0+x_i^T\beta),\qquad \lambda_{ij}=\alpha_i\beta_j\gamma^{I(home)}.
$$

<div dir="rtl">
<p>ב-GLM מקדם חיובי מגדיל את קצב השערים, למשל יתרון בית קבוע באותו מערך פיצ׳רים. ב-Maher α היא חוזק ההתקפה, β היא נטייה לספוג, ו-γ היא מכפלת הביתיות; כולם חיוביים. נדרשת מגבלת זיהוי </p></div>

$$
\sum\log\alpha=0
$$

<div dir="rtl">. כאן Dixon–Coles הוא קו בסיס Maher עם דעיכה ועדכון יומי; אין תיקון תלויות לתוצאות נמוכות (ρ).
<p>לעולה יש prior קפוא בעוצמה של שישה משחקים, סביב ממוצע ליגת העונה </p></div>

$$
R
$$

<div dir="rtl">:
</div>

$$
\frac{k}{2}\sum_{p\in P}\left[(\log\alpha_p-\overline{\log\alpha}_R-a^*)^2+(\log\beta_p-\overline{\log\beta}_R-b^*)^2\right].
$$

<div dir="rtl">
<p>עוצמת הקנס בפועל היא </p></div>

$$
k=6\bar y_w w_0
$$

<div dir="rtl">: ממוצע שערי העבר המשוקלל מתרגם את ששת המשחקים לעוצמה, ו-</div>

$$
w_0
$$

<div dir="rtl"> הוא משקל תחילת העונה ביחס לבוקר החיזוי (1 ללא דעיכה). כך גם המידע המוקדם מתיישן עם העונה.<p>כשיש כמה עולות, נגזרת ממוצע R קושרת את הקנסות. עולה ללא משחקים אינה חייבת להיות בדיוק a*/b*; תנאי האופטימום הוא </p></div>

$$
d_p=\sum_{q\in P}d_q/|R|
$$

<div dir="rtl">. הקנס אינו נכלל ב-NLL הוולידציה.
</div>

In [ ]:
prior = views.frozen_prior()
table(
    pd.DataFrame([prior]),
    {
        "a_star": "a* — התקפה יחסית בלוג",
        "b_star": "b* — ספיגה יחסית בלוג",
        "source_sha256": "חתימת מקור",
        "team_seasons": "קבוצות־עונות",
    },
    precision={"a_star": 6, "b_star": 6},
)

<div dir="rtl"><h2>4 — הרשת קבועה מראש: דיוק מול מורכבות ורגולריזציה</h2>
<p>היפר-פרמטר הוא בחירה שנקבעת מחוץ ללמידת המקדמים: כמה עמוק העץ, כמה תיקונים וכמה קנס. הרשת כוללת ערכים פשוטים ומורכבים, והבחירה לפי ולידציה עתידית. הטבלה נגזרת ישירות מה-YAML. ערך None בזמן מחצית פירושו בלי דעיכה. ברירת מחדל של ספרייה היא מועמד עצמאי מלא. הגרפים שישוו צירופים יתווספו ב-E2c; אין כאן ציוני מועמדים.</p>
</div>

In [ ]:
table(
    views.hyperparameter_table(),
    {
        "model": "מודל ושלב",
        "parameter": "פרמטר",
        "values": "ערכים שנקבעו מראש",
        "meaning": "משמעות בכדורגל",
        "rationale": "הנימוק",
    },
)

<div dir="rtl"><h2>4.1 — loss האימון מספר איך המודל לומד את העבר</h2><p>גמישות גדולה יותר מאפשרת להתאים טוב יותר את האימון: באופטימום מדויק של מחלקות מודלים מקוננות וללא שינוי קנס, loss האימון המינימלי אינו עולה. בפועל, דגימה אקראית, רגולריזציה, קצב למידה ואופטימיזציה סופית אינם מבטיחים ירידה בכל השוואת צירופים. רק ולידציה על עתיד עונה אם המורכבות עוזרת לחיזוי.</p><p>הטבלה נגזרת מיומני אימון חתומים, לא מאימון חוזר. train_loss נשמר float32. ב-Maher יש שורה לכל אימון יומי עם איטרציות, loss סופי והתכנסות; העקומה המקורית נשמרת ביומן הדחוס. GLM שהתכנס באפס איטרציות מופיע עם loss חסר, ולא אפס מומצא.</p></div>

In [ ]:
from ligat.evaluate.goals_experiment import load_goal_experiment
from ligat.viz import goal_training_curves as curves

spec = load_goal_experiment()
status = views.pilot_status()
table(
    status.groupby("status").size().rename("units").reset_index(),
    {"status": "מצב C5", "units": "יחידות"},
)
if status.status.eq("complete").all():
    pilot_training = curves.training_curves(spec, stage="pilot")
    table(
        pd.DataFrame(
            [
                dict(
                    rows=len(pilot_training),
                    units=pilot_training.unit_id.nunique(),
                    bytes=(spec.output / "pilot/training_curves.parquet").stat().st_size,
                )
            ]
        ),
        {"rows": "שורות עקומה", "units": "יחידות", "bytes": "גודל בבתים"},
    )
    for name, figure in curves.pilot_figures(pilot_training).items():
        show(figure, "E2c_" + name)
else:
    display(HTML('<div dir="rtl">היחידות חסרות או לא עדכניות; גרפי האימון אינם נטענים.</div>'))

<div dir="rtl"><h2>4.2 — שלושה קיפולי פיילוט מראים מסלול, ולא הוכחת עדיפות</h2><p>עקומות XGBoost מציגות חציון ו-IQR בין שלושת קיפולי הפיילוט: 1,000 עצים ו-min_child_weight=10, צבע לפי עומק וחלונית לפי קצב למידה. ציר העצים לוגריתמי. מפת האימון מקבעת 1,000 עצים ומציגה חציון לכל עומק וזהירות; n=3 בכל תא, פיזור תיאורי בלבד, ללא מסקנה מובהקת. שאר הצירופים נשמרים בטבלה המלאה.</p><p>ב-CatBoost משווים ברירת מחדל לשלושה עומקי רשת עם קנס 3, חציון ו-IQR על שלוש עונות. loss מקורי של ספרייה אחת אינו בר השוואה לזה של אחרת: CatBoost, GLM ו-Maher עשויים להשמיט log(y!) או לכלול משקל וקנס. אין להסיק איכות חיזוי מהפרש בין הגרפים שלהם. גרף Maher מציג את כל האימונים היומיים: זו מדידת הנדסה, ללא אומדן אוכלוסייה או רווח סמך.</p></div>

<div dir="rtl"><h2>4.3 — טיוטה, לפני QA של Claude ובחירת עמית</h2>
<p><b>אימון מול ולידציה:</b> המפות קוראות את תוצרי full החתומים לאותן תצורות S2-A: 1,000 עצים וקצב 0.1. באימון חציון ו-IQR, ובוולידציה ממוצע משוקלל משחקים ו-CI95 בין קיפולים, n=15 בכל תא. שני הצירים מציגים ערכי רשת בדידים. משקל האימון והרכב העונות שונים מהוולידציה, ולכן הפער אינו אומדן נקי יחיד להתאמת יתר; אבחון E2d משתמש ב-NLL לא משוקלל לשניהם.</p>
<p><b>מפת ההיפר-פרמטרים:</b> אחוז מתרומת Elo הוא 100×(NLL_R0−NLL_candidate)/(NLL_R0−NLL_R1), על אותם 15 קיפולים, עם CI95 בריחוף. 100% פירושו התרומה המלאה של R1 מול R0; ערך שלילי גרוע מ-R0. ברשת המוצגת (1,000 עצים, קצב 0.1), עומק 2 וזהירות 30 נותנים את ה-NLL הנמוך ביותר. זה אינו בוחר תצורה חדשה בסולם.</p>
<p><b>השפעות הפרמטרים:</b> מוצג ממוצע ±SE בין תצורות הרשת, לכל ערך, בנפרד לתכנון A ולתכנון B. n התצורות נמצא בריחוף. זה תיאור של הרשת ולא אפקט סיבתי או דגימות עצמאיות; כש-n=1 ה-SE אינו מוגדר ואינו מוצג. ברירת המחדל נשארת תצורה נפרדת, ולכן נקודה שבה רק היא מיוצגת אינה השוואה מבוקרת.</p>
<p><b>התקדמות הסולם:</b> מוצג הפרש מזווג מול S2, עם ±SE ו-CI95 בריחוף, על n=14 קיפולים עמוקים. השלבים הם תוצאות הסולם המצטבר, ולא ניסוי נפרד בכל משפחה; רק F5 עברה את כלל הכניסה. התוצאות והאבחנות תלויות בהכרעות כלל 6 הפתוחות. אין כאן בחירת מודל ל-M3.</p></div>

In [ ]:
if (spec.output / "full/ladder_report.json").exists():
    frozen_report = curves.frozen_full_report(spec)
    full_training = curves.training_curves(spec, stage="full")
    full_validation = curves.full_validation_table(spec, stage="full")
    for name, figure in curves.full_figures(full_training, full_validation, frozen_report).items():
        show(figure, "E2c_" + name)
else:
    display(HTML('<div dir="rtl">ממתין לדוח D3 חתום; לא מחושבים ציוני מועמדים.</div>'))

<div dir="rtl"><h2>5 — הסולם בודק מידע חדש מול אותו יריב להשוואה</h2>
<p>סולם מוסיף משפחת מידע בכל פעם, ומפריד בין תרומת הפיצ׳ר לבין כוונון המודל. S1 ו-S2 מכווננים ב-A ואחר כך B. בדיקת כניסה משתמשת בתצורת S2 קבועה ובאותם משחקים, עם ובלי המשפחה. במסלול המקוצר דירוג שלבים נוספים מוגבל לחמש תצורות S2, ללא רשת חדשה.</p>
<p>לכל עונה עמוקה מחשבים הפרש מזווג </p></div>

$$
d_s=L_{reference,s}-L_{candidate,s}
$$

<div dir="rtl">, חיובי משמעו שיפור:
</div>

$$
\bar d={\sum_s n_s d_s\over\sum_s n_s},\quad SE={sd(d_s)\over\sqrt n},\quad CI_{95\%}=\bar d\pm t_{n-1,.975}SE.
$$

<div dir="rtl">
<p>n_s הוא מספר המשחקים בעונה ו-n הוא מספר העונות העמוקות. הממוצע משוקלל במשחקים, ופיזור העונות לחישוב SE אינו משוקלל. כלל 8 בגרסה 2 דורש ממוצע שיפור מעל SE, שיפור בלפחות שלוש מחמש העונות האחרונות, וממוצע שיפור חיובי בחמש האחרונות. רווח הסמך מציג אי-ודאות בין עונות, ולא משנה את שער ההחלטה. למשל, תוספת כושר צריכה לשפר חיזוי של אותן קבוצות באותם משחקים, ולא רק לקבל עונות קלות יותר.</p>
<p>כלל 4 מציע מודל פשוט בתוך SE אחת מהמוביל; כלל 6 מסמן כשהמובילות בעמוקים ובחמש האחרונות שונות; כרגע ההרצה ממשיכה באופן זמני עם הצעת כלל 4, ושומרת open_reviews ותלויות. עמית יכריע אחרי QA. כלל 10 מסמן אי-הסכמה עם RPS. F1-R ו-F1-A הן חלופות; אם שתיהן עוברות מוצעת בעלת השיפור הגדול יותר, עם אפשרות להכרעה מפורשת.</p>
</div>

In [ ]:
table(views.ladder_plan_table(), {"stage": "שלב", "features": "עמודות", "meaning": "המידע שנבדק"})

<div dir="rtl"><h2>5.1 — מידע דומה מיוצג פעם אחת</h2>
<p>מסנן Pearson מחושב פעם אחת על פיצ׳רים מספריים ובוליאניים בלבד בעונות הפיתוח. מזהים ו-stage אינם מקבלים מספר סדר מלאכותי. r בטווח [−1,1]; כאן r≈0.957, ולכן שני ההפרשים כמעט מספרים אותו סיפור על העונה הקודמת. שתי ההצעות והכרעת עמית מוצגות בלי לייחס למתאם סיבתיות. ספירת הנתונים היא 8,234 שורות תקפות; זו בדיקת חפיפה תיאורית, לא הוכחת תועלת בחיזוי.</p>
</div>

In [ ]:
table(
    views.correlation_decision_table(),
    {
        "pair": "הזוג",
        "r": "מתאם",
        "codex": "הצעת Codex",
        "claude": "הצעת Claude",
        "amit": "הכרעת עמית",
        "scope": "טווח וחוזה",
    },
    precision={"r": 6},
)

<div dir="rtl"><h2>6 — טיוטה, לפני QA של Claude ובחירת עמית</h2><p>הטבלאות קוראות את ladder_report.json החתום. 15 הקיפולים נותנים תמונת פיתוח מלאה; הקיפולים העמוקים משמשים הכרעה; חמש האחרונות בודקות את הליגה העדכנית. סמני כלל 6 ותלויותיהם נשארים פתוחים לעמית. הסרגלים הם שחזור, ואין כאן בחירת מודל ל-M3.</p><p>הצעת S2 היא תצורה פשוטה בקבוצת הקרובות לפי כלל 4, ואינה בהכרח המובילה העמוקה או בחמש האחרונות. ההצעה, התלויות והאבחון כולם זמניים. אין כאן הכרעה חדשה של עמית.</p></div>

In [ ]:
import re

frozen_report = curves.frozen_full_report(spec)
summary_rows = []
for name, model in frozen_report["readable_metrics"].items():
    for row in model["summary"]:
        summary_rows.append(
            dict(
                model=name,
                label=model["label"],
                scope=row["scope"],
                nll=row["poisson_nll"],
                n=row["rows"],
                probability=row["observed_probability_percent"],
                elo=row["elo_share_percent"],
                non_tie=row["better_non_tie_percent"],
                better=row["better_percent"],
                worse=row["worse_percent"],
                tie=row["tie_percent"],
                mae=row["mae"],
                rmse=row["rmse"],
                provisional=", ".join(model["provisional_on"]),
            )
        )
summary_frame = pd.DataFrame(summary_rows)
scope_names = {"all": "15 הקיפולים", "deep": "קיפולים עמוקים", "last5": "חמש האחרונות"}
table(
    summary_frame[
        ["model", "label", "scope", "nll", "probability", "elo", "non_tie", "n", "provisional"]
    ],
    {
        "model": "שלב או קו בסיס",
        "label": "תצורה קריאה",
        "scope": "טווח",
        "nll": "NLL",
        "n": "שורות קבוצה במשחק",
        "probability": "הסתברות גאומטרית באחוזים",
        "elo": "אחוז מתרומת Elo",
        "non_tie": "אחוז שורות עדיפות מבין הלא שוות מול S2",
        "provisional": "תלוי בהכרעות",
    },
    values=scope_names,
    precision={"nll": 6, "probability": 1, "elo": 1, "non_tie": 1},
)
table(
    summary_frame[["model", "scope", "mae", "rmse", "better", "worse", "tie"]],
    {
        "model": "שלב או קו בסיס",
        "scope": "טווח",
        "mae": "MAE בשערים",
        "rmse": "RMSE בשערים",
        "better": "אחוז טוב יותר מול S2",
        "worse": "אחוז גרוע יותר",
        "tie": "אחוז שווה",
    },
    values=scope_names,
)
paired_rows = []
for name, model in frozen_report["readable_metrics"].items():
    stats = model["paired"]
    paired_rows.append(
        dict(
            model=name,
            improvement=stats["mean_improvement"],
            se=stats["se"],
            lower=stats["ci_lower"],
            upper=stats["ci_upper"],
            n=stats["n"],
        )
    )
table(
    pd.DataFrame(paired_rows),
    {
        "model": "שלב או קו בסיס",
        "improvement": "שיפור מזווג מול S2 בעמוקים",
        "se": "SE",
        "lower": "CI95 תחתון",
        "upper": "CI95 עליון",
        "n": "עונות עמוקות משותפות",
    },
    precision={"improvement": 6, "se": 6, "lower": 6, "upper": 6, "recent_improvement": 6},
    intervals={"improvement": ("lower", "upper")},
)
entries = []
for decision in frozen_report["decisions"]:
    if decision["kind"] == "entry":
        stats = decision["rule8"]["stats"]
        actual_entry = (
            decision["name"] in frozen_report["entered"]
            if decision["name"] in ("F1-R", "F1-A")
            else decision["entered"]
        )
        entries.append(
            dict(
                name=decision["name"],
                improvement=stats["mean_improvement"],
                se=stats["se"],
                lower=stats["ci_lower"],
                upper=stats["ci_upper"],
                n=stats["n"],
                recent_improvement=stats["last5_improvement"],
                recent_better=stats["last5_folds_better"],
                rule10=decision["rule10"],
                passes=decision["rule8"]["passes"],
                entered=actual_entry,
            )
        )
table(
    pd.DataFrame(entries),
    {
        "name": "בדיקת בסיס או משפחה",
        "improvement": "שיפור מזווג",
        "se": "SE",
        "lower": "CI95 תחתון",
        "upper": "CI95 עליון",
        "n": "עונות עמוקות",
        "recent_improvement": "שיפור בחמש האחרונות",
        "recent_better": "עונות עדיפות מתוך 5",
        "rule10": "דגל כלל 10",
        "passes": "עברה כלל 8",
        "entered": "נכנסה אוטומטית",
    },
    precision={"improvement": 6, "se": 6, "lower": 6, "upper": 6, "recent_improvement": 6},
    intervals={"improvement": ("lower", "upper")},
)
flag_rows = [
    dict(stage=d["name"], kind=d["kind"])
    for d in frozen_report["decisions"]
    if d.get("rule10") is True
]
table(
    pd.DataFrame(flag_rows),
    {"stage": "שלב עם דגל כלל 10", "kind": "סוג הבדיקה"},
    values={"selection": "בחירה בין תצורות", "entry": "כניסת מידע"},
)
display(HTML('<div dir="rtl"><b>open_reviews — הכרעות עמית שעדיין נדרשות</b></div>'))
review_rows = []
for review in frozen_report["open_reviews"]:
    decision = next(d for d in frozen_report["decisions"] if d["name"] == review["name"])
    review_rows.append(
        dict(
            stage=review["name"],
            deep="; ".join(x["label"] for x in review["deep_leaders"]),
            recent="; ".join(x["label"] for x in review["recent_leaders"]),
            proposal=decision["labels"][review["proposal"]],
            dependents=", ".join(
                dict.fromkeys(re.sub(r"[0-9a-f]{128}$", "", name) for name in review["dependents"])
            ),
        )
    )
table(
    pd.DataFrame(review_rows),
    {
        "stage": "שלב",
        "deep": "מובילה עמוקה",
        "recent": "מובילה בחמש האחרונות",
        "proposal": "הצעה זמנית שבה ממשיכים",
        "dependents": "תלויים בהכרעה",
    },
)

<div dir="rtl"><h2>6.1 — אבחון מאוחר מראה היכן הוולידציה מפסיקה להשתפר</h2><p>אחרי הקפאת ההכרעות אומנו מחדש זוכת S2 ושכנותיה המיידיות בעומק ובקצב, וכן מספר העצים המרבי, ב-15 קיפולים. כל יתר ההגדרות קבועות. בקצה הרשת אין ממציאים שכן מבחוץ; תצורות זהות מוסרות. העקומות מציגות חציון ו-IQR, n=15, של NLL לא משוקלל באימון ובוולידציה; משקלי אופטימיזציית האימון נשארו אלה שנבחרו.</p><p>הקו האנכי מסמן את מספר העצים שנבחר לפני האבחון. כשהוולידציה מתייצבת או עולה בזמן שהאימון ממשיך לרדת, זו עדות אפשרית להתאמת יתר. זה אינו early stopping, אינו בוחר מספר עצים, ואינו משנה שום הכרעה. קובצי האבחון נפרדים, חתומים ומסומנים diagnostic=true.</p><p><b>על הבחירה הזמנית; אם עמית יכריע אחרת, האבחון ירוץ שוב.</b> בהצעה הנוכחית, עומק 2 וקצב 0.05 הם גבולות תחתונים: שכנים מחוץ לרשת דולגו. האבחון כולל ארבע תצורות ו-60 אימונים, והשכנה עם 1,000 עצים נשמרת בנפרד מן ההצעה עם 150 עצים.</p><p>בעקומת 1,000 העצים (עומק 2, קצב 0.05), חציון NLL הוולידציה כמעט מתייצב בין 150 ל-400 עצים (כ-1.418 בשתי הנקודות), ועולה לכ-1.423 ב-1,000; חציון האימון ממשיך לרדת מכ-1.399 ב-150 לכ-1.368 ב-1,000. בקצב 0.1 חציון הוולידציה כמעט זהה ב-100 וב-150 עצים (כ-1.4172). בהצעת 150 העצים עם קצב 0.05 הוולידציה עדיין יורדת בין 100 ל-150. אלה תיאורי חציון ו-IQR של 15 קיפולים, לא הוכחת מובהקות או המלצה לנקודת עצירה; 150 העצים המקוריים והכרעות D3 נשארים כפי שהיו.</p></div>

In [ ]:
diagnostic = curves.load_learning_diagnostics(spec)
show(curves.learning_curve_figure(diagnostic), "E2d_learning_curves_train_vs_validation")
table(
    pd.DataFrame(
        [
            {
                key: diagnostic[key]
                for key in (
                    "configs",
                    "folds",
                    "files",
                    "frozen_artifacts",
                    "frozen_artifacts_unchanged",
                )
            }
        ]
    ),
    {
        "configs": "תצורות אבחון",
        "folds": "קיפולים",
        "files": "קובצי אבחון",
        "frozen_artifacts": "קובצי D3 שנבדקו",
        "frozen_artifacts_unchanged": "D3 נשמר ללא שינוי",
    },
)

<div dir="rtl"><h2>7 — להבין את המודל</h2>
<p>ממתין לסבב הבא. בפרקי התוצאות: ממתין ל-D3, ל-QA ולהכרעת עמית; בפרק ההסבר יופיעו כיול, שאריות וביתיות לאחר בחירת המודל.</p>
</div>

<div dir="rtl"><h2>8 — הנדסה שומרת על ניסוי עיוור, סיבתי וניתן לחידוש</h2>
<p>pilot שומר λ ו-loss אימון בלבד; אין בו יעד ולידציה או NLL, RPS, log loss או Brier. כל יחידה ויומן חתומים, והאינדקס הוא תצוגה נגזרת. חסר או לא עדכני מוצגים במפורש, בלי דריסה או אימון אוטומטי במחברת. חתימת היחידה מכסה 37 תלויות מפורשות וגרסאות ספריות; קוד תצוגה אינו מבטל יחידות. המאמן אינו כותב בכל איטרציה: היומן נאסף בזיכרון ונכתב פעם אחת בסיום.</p><p>Maher נבדק ב-11,600 אימונים יומיים, ללא כשלים. בטבלאות הבאות זמני מפקד הפיילוט: חציון וממוצע ליחידה, ולא אומדן אוכלוסייה. מדידת עובדים היא הרצה אחת לכל מספר, n=60 יחידות קבועות; אין כאן רווח סמך של תזמון חוזר.</p>
</div>

In [ ]:
engineering = views.engineering_tables()
table(
    engineering["backend"],
    {
        "timing_group": "קבוצת תזמון",
        "units": "יחידות",
        "fit_mean": "ממוצע D2 בשניות",
        "fit_median": "חציון D2 בשניות",
        "fit_mean_D2b": "ממוצע D2b בשניות",
        "fit_median_D2b": "חציון D2b בשניות",
        "mean_change_percent": "שינוי בממוצע באחוזים",
    },
)
table(
    engineering["wall"],
    {
        "run": "הרצה",
        "seconds": "זמן CLI בשניות",
        "calibration": "מקדם כיול",
        "rss": "שיא RSS כולל ב-MiB",
    },
)
table(
    engineering["bench"],
    {"n_jobs": "תהליכים", "wall_seconds": "זמן קיר בשניות", "units_per_hour": "יחידות בשעה"},
)

<div dir="rtl"><h2>8.1 — ציון טוב מדי דורש בדיקת דליפה</h2>
<p>log loss של בית/תיקו/חוץ נגזר ממטריצת 6×6: שערים 0–4 ו-5+, ותא 5+:5+ נחשב תיקו לפי ה-ש29. הטווח הוא [0,∞), ופחות טוב יותר. ממוצע תצורה מתחת ל-0.98 עוצר כחשד לדליפה; קיפול בודד רק מסומן. כל תצורה מתחת ל-1.00 מסומנת ל-QA. שלוש התוצאות מסודרות בית/תיקו/חוץ; RPS בטווח [0,1] מתחשב גם בסדר הזה. ״משחקים שנראו מול שלא נראו״ ממתין לבדיקה לאחר בחירת המודל.</p>
</div>

<div dir="rtl"><h2>9 — M3 יקבל את מדיניות האימון שנבדקה בפועל</h2>
<p>עמית בחר as_evaluated: אותו unit_id ואותה היסטוריה כמו בהערכה. למשל, Maher יומי משתמש גם בתוצאות מוקדמות בעונה הנוכחית, בתנאי train_end_date &lt; as_of. המדיניות החלופית prior_seasons_only מחריגה את העונה הנוכחית ומחייבת יחידות אחרות. אין ברירת מחדל; המדיניות היא חלק מחתימת השכבה. train_end_season מתאר עונה שלמה, ולכן תמיד קטן מעונת היעד, גם כשהתאריך שייך לעונה הנוכחית.</p><p>היצוא במצב as_evaluated קורא יחידות full שהושלמו, בלי לאמן חסרות ובלי לשנות את יחידות D3. Parquet ו-manifest נחתמים עם קלט, קוד, תצורה ופלט. חיבור שתי שורות המשחק נותן lambda_home ו-lambda_away. השכבה האמיתית ממתינה ל-QA ולבחירת עמית.</p>
</div>

In [ ]:
table(views.lambda_schema_table(), {"column": "עמודה", "meaning": "משמעות וחוזה"})

<div dir="rtl"><h2>10 — מילון: המספרים צריכים להתפרש בכדורגל</h2>
<p><b>λ:</b> ממוצע שערים חיובי; 1.6 אינו הבטחה לשני שערים. <b>Poisson:</b> התפלגות ספירה עם ממוצע ושונות λ; הסתברויות בין אפס לאחד. <b>NLL:</b> הפתעה מהתוצאה שהתרחשה, [0,∞); חיזוי 0.05 לתוצאה שהתרחשה נענש יותר מחיזוי 0.4. <b>Deviance:</b> הפסד ביחס להתאמה מושלמת, [0,∞); אין להשוות גולמית ל-NLL של ספרייה אחרת.</p>
<p><b>סולם:</b> הוספת סוג מידע מול אותו בסיס; למשל האם כושר מוסיף מעבר לכוח וביתיות. <b>קיפול:</b> עונה עתידית נפרדת מהיסטוריית האימון. <b>OOF:</b> תחזית שלא למדה את תוצאת המשחק, גם בעדכון יומי. <b>עקומת למידה:</b> שינוי loss לאורך הוספת עצים; ציר אחד הוא איטרציות והאחר הפסד. <b>Overfitting:</b> למידת פרטי עבר שלא עוזרים במשחק הבא; loss אימון נמוך עם ולידציה שאינה משתפרת הוא סימן.</p>
<p><b>SE:</b> שגיאת תקן לאומדן, לא טעות למשחק בודד; יחידותיו כיחידות המדד. <b>IQR:</b> התחום בין הרבעונים, המכיל את 50% המרכזיים; למשל פיזור loss בין שלוש עונות פיילוט הוא תיאורי בלבד. <b>רווח סמך:</b> טווח אי-ודאות לאומדן לפי יחידת הדגימה; כאן עונות מזווגות, לא שורות משחק עצמאיות. <b>רגולריזציה:</b> ריסון מורכבות באימון, לא תוספת למדד הבחירה. <b>זמן מחצית:</b> גיל שבו משקל משחק יורד לחצי. <b>Prior:</b> הנחה מוקדמת לעולה בהיעדר היסטוריה מספקת.</p>
<p><b>כיול:</b> האם קבוצות שחוזים להן 1.8 מבקיעות בממוצע 1.8. <b>RSS:</b> זיכרון פיזי במשתמשי ההרצה, ב-MiB. <b>Cache וחתימה:</b> שימוש חוזר רק בתוצר שהקלט, הקוד והתצורה שלו תואמים. <b>מצב זמני:</b> חישובים תלויים בהכרעה שעמית עוד לא אישר; אין לראות בהם בחירה סופית.</p>
<p><b>MAE:</b> ממוצע גודל הטעות בשערים, בטווח [0,∞); אם נחזו 1.6 והובקע שער אחד, גודל הטעות הוא 0.6. <b>RMSE:</b> שורש ממוצע ריבועי הטעויות, באותן יחידות ובאותו טווח; החמצה של ארבעה שערים משפיעה יותר מאשר ב-MAE. שניהם מלווים את NLL ואינם מכריעים את הבחירה.</p>
<p><b>קיפול עמוק:</b> לפחות ארבע עונות אימון קודמות. מספר העונות העמוקות יכול להיות שונה בין backends, וההשוואה המזווגת משתמשת בעונות המשותפות בלבד. n של רווח הסמך הוא מספר העונות, ולצידו מוצג גם מספר שורות הקבוצה במשחק.</p>
<p><b>הסיכום ממתין:</b> אין עדיין מודל שנבחר ל-M3, אין שכבת λ אמיתית, ופרק 0 נשאר פתוח עד QA ובחירת עמית.</p>
</div>